# Projection and Predicate pushdown in Apache Parquet using Spark


In [2]:
%run ../start.ipynb

Args: Namespace(data_format='none', port_offset=2) - unknown_args: ['-f', '/home/jovyan/.local/share/jupyter/runtime/kernel-cc46f7d7-31ce-4976-ac6a-316ee328b881.json']
Spark version: 4.1.2, Driver memory: 16g, Executor memory: 8g, Service: jupyter-spark-4.1, Data format: None
Spark packages: 
Spark extensions: 
Spark catalog configs: {}
+-------------+
|      catalog|
+-------------+
|spark_catalog|
+-------------+



Version,4.1.2
Master,local[2]
AppName,main


               total        used        free      shared  buff/cache   available
Mem:            62Gi       8.6Gi        45Gi       292Mi       9.5Gi        53Gi
Swap:          8.0Gi          0B       8.0Gi


In [3]:
path="/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset"
!ls -la {path}

total 5799848
drwxr-xr-x. 1 jovyan users        332 Jul 15 20:42  .
drwxr-xr-x. 1 jovyan  1000         62 Jul 15 20:37  ..
drwxr-xr-x. 1 jovyan users         16 Jul 15 20:39  .complete
-rw-r--r--. 1 jovyan users       5356 Jul 15 20:38  female_coaches.csv
-rw-r--r--. 1 jovyan users    1685124 Jul 15 20:38 'female_players (legacy).csv'
-rw-r--r--. 1 jovyan users   94212088 Jul 15 20:38  female_players.csv
-rw-r--r--. 1 jovyan users    2214250 Jul 15 20:38  female_teams.csv
-rw-r--r--. 1 jovyan users     132879 Jul 15 20:38  male_coaches.csv
-rw-r--r--. 1 jovyan users   90933390 Jul 15 20:38 'male_players (legacy).csv'
-rw-r--r--. 1 jovyan users 5637100640 Jul 15 20:39  male_players.csv
-rw-r--r--. 1 jovyan users  112744779 Jul 15 20:39  male_teams.csv
drwxr-xr-x. 1 jovyan users         24 Jul 15 20:42  parquet


In [4]:
csv_file = Path(path) / "male_players.csv"
parquet_output = Path(path) / "parquet" / "male_players"

In [5]:
%load_ext autotime

time: 28.3 μs (started: 2026-07-21 18:58:58 +00:00)


In [6]:
if not parquet_output.is_dir():
    print(f"Convert CSV to parquet")
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(str(csv_file))
    )

    df.write.mode("overwrite").parquet(str(parquet_output))
 
else:
    print(f"Read parquet")
    
    df = (
        spark.read
        .parquet(str(parquet_output))
    )      

Read parquet
time: 371 ms (started: 2026-07-21 18:58:58 +00:00)


In [7]:
!echo "Number of files: $(find {path}/parquet/male_players/*.parquet | wc -l)"
!echo "Files: $(ls {path}/parquet/male_players/*.parquet)"

Number of files: 42
Files: /home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset/parquet/male_players/part-00000-68039858-fa40-49bb-96ba-e35375d87067-c000.snappy.parquet
/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset/parquet/male_players/part-00001-68039858-fa40-49bb-96ba-e35375d87067-c000.snappy.parquet
/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset/parquet/male_players/part-00002-68039858-fa40-49bb-96ba-e35375d87067-c000.snappy.parquet
/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset/parquet/male_players/part-00003-68039858-fa40-49bb-96ba-e35375d87067-c000.snappy.parquet
/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset/parquet/male_players/part-00004-68039858-fa40-49bb-96ba-e35375d87067-c000.snappy.parquet
/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset/parque

In [8]:
humanize.intword(df.count())

'10.0 million'

time: 446 ms (started: 2026-07-21 18:58:58 +00:00)


In [9]:
df.explain()

== Physical Plan ==
FileScan parquet [player_id#16,player_url#17,fifa_version#18,fifa_update#19,fifa_update_date#20,short_name#21,long_name#22,player_positions#23,overall#24,potential#25,value_eur#26,wage_eur#27,age#28,dob#29,height_cm#30,weight_kg#31,league_id#32,league_name#33,league_level#34,club_team_id#35,club_name#36,club_position#37,club_jersey_number#38,club_loaned_from#39,club_joined_date#40,... 85 more fields] Batched: false, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-co..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<player_id:int,player_url:string,fifa_version:int,fifa_update:int,fifa_update_date:date,sho...


time: 8.35 ms (started: 2026-07-21 18:58:59 +00:00)


# Predicate pushdown

In [10]:
df_all = spark.read.parquet(str(parquet_output))

df_select = (
    spark.read
    .parquet(str(parquet_output))
    .select("value_eur")
)

time: 121 ms (started: 2026-07-21 18:58:59 +00:00)


In [11]:
df_select_filter = (
    spark.read
    .parquet(str(parquet_output))
    .filter(F.col("value_eur") > 1000000)
    .select("value_eur")
)

time: 68.4 ms (started: 2026-07-21 18:58:59 +00:00)


In [12]:
humanize.intword(df_select_filter.count())

'3.7 million'

time: 391 ms (started: 2026-07-21 18:58:59 +00:00)


In [13]:
humanize.intword(df_select.count())

'10.0 million'

time: 120 ms (started: 2026-07-21 18:58:59 +00:00)


In [14]:
viewdf(df_select_filter, limit=5)

,value_eur
0,103500000
1,63000000
2,111000000
3,132000000
4,129000000


time: 67.6 ms (started: 2026-07-21 18:58:59 +00:00)


# Physical plan
See "PushedFilters:"

In [15]:
df_select_filter.explain("formatted")

== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [value_eur#471]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset/parquet/male_players]
PushedFilters: [IsNotNull(value_eur), GreaterThan(value_eur,1000000)]
ReadSchema: struct<value_eur:int>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [value_eur#471]

(3) Filter [codegen id : 1]
Input [1]: [value_eur#471]
Condition : (isnotnull(value_eur#471) AND (value_eur#471 > 1000000))


time: 7.01 ms (started: 2026-07-21 18:59:00 +00:00)


In [16]:
df_select.explain("formatted")

== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [value_eur#360]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset/parquet/male_players]
ReadSchema: struct<value_eur:int>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [value_eur#360]


time: 4.23 ms (started: 2026-07-21 18:59:00 +00:00)


In [17]:
humanize.intword(df_select_filter.count())

'3.7 million'

time: 142 ms (started: 2026-07-21 18:59:00 +00:00)


In [18]:
humanize.intword(df_select.count())

'10.0 million'

time: 85.8 ms (started: 2026-07-21 18:59:00 +00:00)


In [19]:
def benchmark(label, query):
    # Run once to reduce JVM / planning noise
    query.collect()

    start = time.perf_counter()
    result = query.collect()
    elapsed = time.perf_counter() - start

    print(f"{label}: {elapsed:.3f} s")
    #return result

time: 295 μs (started: 2026-07-21 18:59:00 +00:00)


In [20]:
benchmark("Only value_eur", df_select)

Only value_eur: 9.163 s
time: 20.9 s (started: 2026-07-21 18:59:00 +00:00)


In [21]:
df_select_filter.explain("simple")

== Physical Plan ==
*(1) Filter (isnotnull(value_eur#471) AND (value_eur#471 > 1000000))
+- *(1) ColumnarToRow
   +- FileScan parquet [value_eur#471] Batched: true, DataFilters: [isnotnull(value_eur#471), (value_eur#471 > 1000000)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-co..., PartitionFilters: [], PushedFilters: [IsNotNull(value_eur), GreaterThan(value_eur,1000000)], ReadSchema: struct<value_eur:int>


time: 2.26 ms (started: 2026-07-21 18:59:21 +00:00)


In [22]:
df_select_filter.explain("formatted")

== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [value_eur#471]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset/parquet/male_players]
PushedFilters: [IsNotNull(value_eur), GreaterThan(value_eur,1000000)]
ReadSchema: struct<value_eur:int>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [value_eur#471]

(3) Filter [codegen id : 1]
Input [1]: [value_eur#471]
Condition : (isnotnull(value_eur#471) AND (value_eur#471 > 1000000))


time: 5.86 ms (started: 2026-07-21 18:59:21 +00:00)


In [23]:
df_select_filter.explain("extended")

== Parsed Logical Plan ==
'Project ['value_eur]
+- Filter (value_eur#471 > 1000000)
   +- Relation [player_id#461,player_url#462,fifa_version#463,fifa_update#464,fifa_update_date#465,short_name#466,long_name#467,player_positions#468,overall#469,potential#470,value_eur#471,wage_eur#472,age#473,dob#474,height_cm#475,weight_kg#476,league_id#477,league_name#478,league_level#479,club_team_id#480,club_name#481,club_position#482,club_jersey_number#483,club_loaned_from#484,club_joined_date#485,... 85 more fields] parquet

== Analyzed Logical Plan ==
value_eur: int
Project [value_eur#471]
+- Filter (value_eur#471 > 1000000)
   +- Relation [player_id#461,player_url#462,fifa_version#463,fifa_update#464,fifa_update_date#465,short_name#466,long_name#467,player_positions#468,overall#469,potential#470,value_eur#471,wage_eur#472,age#473,dob#474,height_cm#475,weight_kg#476,league_id#477,league_name#478,league_level#479,club_team_id#480,club_name#481,club_position#482,club_jersey_number#483,club_loaned_

In [24]:
df_select_filter.explain("codegen")

Found 1 WholeStageCodegen subtrees.
== Subtree 1 / 1 (maxMethodCodeSize:255; maxConstantPoolSize:140(0.21% used); numInnerClasses:0) ==
*(1) Filter (isnotnull(value_eur#471) AND (value_eur#471 > 1000000))
+- *(1) ColumnarToRow
   +- FileScan parquet [value_eur#471] Batched: true, DataFilters: [isnotnull(value_eur#471), (value_eur#471 > 1000000)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-co..., PartitionFilters: [], PushedFilters: [IsNotNull(value_eur), GreaterThan(value_eur,1000000)], ReadSchema: struct<value_eur:int>

Generated code:
/* 001 */ public Object generate(Object[] references) {
/* 002 */   return new GeneratedIteratorForCodegenStage1(references);
/* 003 */ }
/* 004 */
/* 005 */ // codegenStageId=1
/* 006 */ final class GeneratedIteratorForCodegenStage1 extends org.apache.spark.sql.execution.BufferedRowIterator {
/* 007 */   private Object[] references;
/* 008 */   private scala.collection.Itera

In [25]:
df_select_filter.explain("cost")

== Optimized Logical Plan ==
Project [value_eur#471], Statistics(sizeInBytes=12.7 MiB)
+- Filter (isnotnull(value_eur#471) AND (value_eur#471 > 1000000)), Statistics(sizeInBytes=1220.7 MiB)
   +- Relation [player_id#461,player_url#462,fifa_version#463,fifa_update#464,fifa_update_date#465,short_name#466,long_name#467,player_positions#468,overall#469,potential#470,value_eur#471,wage_eur#472,age#473,dob#474,height_cm#475,weight_kg#476,league_id#477,league_name#478,league_level#479,club_team_id#480,club_name#481,club_position#482,club_jersey_number#483,club_loaned_from#484,club_joined_date#485,... 85 more fields] parquet, Statistics(sizeInBytes=1220.7 MiB)

== Physical Plan ==
*(1) Filter (isnotnull(value_eur#471) AND (value_eur#471 > 1000000))
+- *(1) ColumnarToRow
   +- FileScan parquet [value_eur#471] Batched: true, DataFilters: [isnotnull(value_eur#471), (value_eur#471 > 1000000)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/kaggle/datasets/stefano

In [26]:
benchmark("Filtered, with predicate pushdown", df_select_filter)

Filtered, with predicate pushdown: 3.565 s
time: 7.51 s (started: 2026-07-21 18:59:21 +00:00)
